# De píxeles a patches: cómo una imagen entra en un Transformer

**Facsímil 11 · IA multimodal y percepción** — capítulo 2 (de píxeles a patches: cómo una imagen se
convierte en representación).

Los Transformers nacieron para texto. ¿Cómo es que hoy entienden imágenes? El truco, sencillo y
brillante, es el que descubres en este cuaderno: una foto no se mete píxel a píxel, se **trocea en
parches**, y cada parche se convierte en un vector, igual que una frase se trocea en *tokens*. Lo haces a
mano, con NumPy, y compruebas que una imagen es, para el modelo, **una secuencia de vectores**. Esa idea
—imagen = secuencia de parches— es la llave de los *Vision Transformers* (ViT) y, sobre ellos, de CLIP,
los modelos visión-lenguaje y medio sistema multimodal.

### Qué vas a aprender
- Por qué un Transformer de texto, casi sin cambios, sirve para imágenes.
- Cómo se **trocea** una imagen en parches y cómo cada parche se **aplana** en un vector.
- Qué papel juegan el **embedding lineal** (la proyección aprendida) y los **embeddings de posición**.
- Por qué sin posición una imagen y su versión **barajada** serían idénticas para el modelo.
- Que trocear **no pierde información**: la imagen se puede reconstruir parche a parche.
- Qué pinta el **token de clase** (`[CLS]`) y cómo la atención mide qué parches se parecen.
- El compromiso **tamaño de parche** ↔ longitud de la secuencia ↔ coste.

### Cómo se usa
Las celdas vienen **sin ejecutar**: pulsa ▶ de arriba abajo y verás nacer cada número y cada gráfico.

### Cuánto cuesta
Unos 18 minutos. CPU, sin claves.


> **Inteligencia artificial para gente curiosa** · facsímil interactivo
> 
> Web del facsímil: https://www.iaparagentecuriosa.686f6c61.dev/ · Autor: @686f6c61 · Fecha: 2026-06-26 · Versión 1.0
> 
> Este cuaderno acompaña al facsímil: ejecútalo de arriba abajo, lee cada celda de texto
> antes de correr la de código y detente en las salidas. La gracia no es que «salga», sino
> entender *por qué* sale.

## 1. El problema: el texto es una secuencia, la imagen una cuadrícula

Un Transformer espera una **secuencia** de vectores (las palabras de una frase). Una imagen, en cambio,
es una cuadrícula 2D de píxeles. No puedes meter los píxeles uno a uno: una imagen modesta de 224×224
tiene más de 50.000 píxeles, y la atención (que compara cada elemento con todos) costaría una fortuna. La
solución del *Vision Transformer*: agrupar los píxeles en **parches** (trozos de, digamos, 16×16) y
tratar cada parche como un «token». Así una imagen se convierte en una secuencia corta y manejable.

Empezamos con una imagen sintética pequeña y reconocible, para ver cada paso con claridad.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)

# Imagen sintetica reconocible (64x64, color): fondo claro, un circulo oscuro y un cuadrado azul.
H = W = 64
img = np.ones((H, W, 3)) * 0.95
yy, xx = np.mgrid[0:H, 0:W]
img[(yy-20)**2 + (xx-20)**2 < 13**2] = [0.15, 0.15, 0.15]   # circulo oscuro
img[38:56, 36:56] = [0.10, 0.45, 0.85]                       # cuadrado azul
print(f"Imagen: {img.shape[0]}x{img.shape[1]} pixeles, {img.shape[2]} canales (R, G, B).")
print(f"Eso son {img.size} numeros en una cuadricula. Demasiados para meterlos uno a uno.")
plt.figure(figsize=(3,3)); plt.imshow(img); plt.title("La imagen (64x64)"); plt.axis("off"); plt.show()


## 2. Trocear en parches

Cortamos la imagen en una rejilla de parches de 16×16 píxeles. Una imagen de 64×64 da 4×4 = **16
parches**. Cada parche es un trocito de imagen; en cuanto los pongamos en fila, serán nuestra
«secuencia», como las palabras de una frase. El orden es el de lectura: izquierda a derecha, arriba abajo.


In [ ]:
P = 16
n = H // P                                  # parches por lado (4)
parches = img.reshape(n, P, W//P, P, 3).swapaxes(1, 2).reshape(n*n, P, P, 3)
print(f"La imagen {H}x{W} se ha partido en {parches.shape[0]} parches de {P}x{P} pixeles.")
print(f"Cada parche es un array {parches.shape[1]}x{parches.shape[2]}x{parches.shape[3]}.")
fig, axes = plt.subplots(n, n, figsize=(4,4))
for k, ax in enumerate(axes.flat):
    ax.imshow(parches[k]); ax.axis("off")
    ax.set_title(str(k), fontsize=7)
fig.suptitle(f"Los {n*n} parches, numerados en orden de lectura"); plt.tight_layout(); plt.show()


## 3. La imagen como una tira: parche a parche

Antes de convertir cada parche en números, conviene **ver** que una imagen ya es una secuencia. Resumimos
cada parche por su color medio y los ponemos en una sola fila: esa tira de 16 casillas es el orden en que
el modelo «lee» la imagen. Lo que para nosotros es una cuadrícula, para el Transformer es esta fila.


In [ ]:
medios = parches.reshape(n*n, -1, 3).mean(axis=1)        # color medio de cada parche -> (16, 3)
plt.figure(figsize=(8, 1.4))
plt.imshow(medios.reshape(1, n*n, 3), aspect="auto")
plt.yticks([]); plt.xticks(range(n*n))
plt.title("La imagen leida como una secuencia de 16 parches (color medio)"); plt.show()
print("Parche  3 (esquina sup-der, fondo)  color medio RGB:", np.round(medios[3], 2))
print("Parche  5 (zona del circulo oscuro) color medio RGB:", np.round(medios[5], 2))
print("Parche 14 (zona del cuadrado azul)  color medio RGB:", np.round(medios[14], 2))
print("Tres parches, tres 'palabras' distintas: ahi vive la informacion.")


## 4. Cada parche, un vector

Aplanamos cada parche: 16×16 píxeles × 3 colores = **768 números**. La imagen entera deja de ser una
cuadrícula de píxeles y pasa a ser una **secuencia de 16 vectores de 768**. Eso es exactamente lo que
recibe un Transformer: una secuencia, igual que con el texto (donde cada *token* es un vector).


In [ ]:
secuencia = parches.reshape(n*n, -1)
print("Forma de la secuencia (parches x dimensiones):", secuencia.shape)
print(f"La imagen es ahora una SECUENCIA de {secuencia.shape[0]} vectores de {secuencia.shape[1]} numeros.\n")
print("Parche 0 (fondo claro),    primeros 6 numeros:", np.round(secuencia[0][:6], 2))
print("Parche 5 (circulo oscuro), primeros 6 numeros:", np.round(secuencia[5][:6], 2))
print("Parche 14 (cuadrado azul), primeros 6 numeros:", np.round(secuencia[14][:6], 2))
print("\nSus vectores son distintos: ahi esta la informacion que el modelo usara para distinguir formas.")


## 5. Trocear no pierde información: reconstruir la imagen

Una duda razonable: al trocear, ¿no se «rompe» la imagen? No. La partición es **reversible**: con los 16
parches y su orden, recomponemos la imagen original píxel a píxel. Lo comprobamos midiendo el error entre
la original y la reconstruida (debe ser exactamente cero).


In [ ]:
recon = parches.reshape(n, n, P, P, 3).swapaxes(1, 2).reshape(H, W, 3)
error = float(np.abs(recon - img).max())
print(f"Error maximo entre la imagen original y la reconstruida desde parches: {error:.6f}")
print("Cero: trocear y volver a montar no pierde ni un pixel. Solo cambia la FORMA, no el contenido.")
fig, ax = plt.subplots(1, 2, figsize=(5, 2.6))
ax[0].imshow(img);   ax[0].set_title("original");      ax[0].axis("off")
ax[1].imshow(recon); ax[1].set_title("desde parches"); ax[1].axis("off")
plt.tight_layout(); plt.show()


## 6. El embedding lineal: una proyección aprendida

En un ViT real, cada vector de parche **no entra tal cual**: pasa antes por una **proyección lineal
aprendida** (una multiplicación por una matriz) que lo lleva a la dimensión de trabajo del modelo, $D$.
Es el equivalente exacto del *embedding* de palabras: convertir el vector crudo en uno que el modelo sabe
manejar. Aquí usamos una matriz aleatoria (en un modelo real se aprende), solo para ver las **formas**:
de 768 pasamos a, digamos, $D = 32$.


In [ ]:
rng = np.random.default_rng(0)
D = 32                                           # dimension de trabajo del modelo (en un ViT real, cientos)
Wproj = rng.standard_normal((secuencia.shape[1], D)) / np.sqrt(secuencia.shape[1])
embeddings = secuencia @ Wproj                   # (16, 768) @ (768, 32) = (16, 32)
print("Vectores crudos:", secuencia.shape, " -> proyeccion lineal -> embeddings:", embeddings.shape)
print(f"Cada parche pasa de {secuencia.shape[1]} numeros a {D}: la 'puerta de entrada' al Transformer.")
print("En un modelo real, Wproj se APRENDE; aqui es aleatoria, solo para ver el cambio de forma.")


## 7. El problema de la posición: barajar los parches

Al poner los parches en fila, **se pierde dónde estaba cada uno**. Y la atención, por sí sola, no sabe de
posiciones: trata la entrada como una **bolsa** de vectores. Lo demostramos de forma tajante: barajamos
los parches. La imagen resultante es un sinsentido para nosotros... pero la **bolsa** de vectores es la
misma (los mismos 16 vectores, en otro orden). Cualquier operación que ignore el orden no las distingue.


In [ ]:
orden = rng.permutation(n*n)
parches_baraj = parches[orden]
secuencia_baraj = secuencia[orden]
img_baraj = parches_baraj.reshape(n, n, P, P, 3).swapaxes(1, 2).reshape(H, W, 3)

# La 'bolsa' de vectores no cambia con el orden: la suma sobre los parches es identica.
misma_bolsa = np.allclose(secuencia.sum(axis=0), secuencia_baraj.sum(axis=0))
print("¿La bolsa de vectores (suma sobre parches) es la misma tras barajar?", misma_bolsa)
print("Para una operacion que ignora el orden, la imagen y su version barajada son INDISTINGUIBLES.")
fig, ax = plt.subplots(1, 2, figsize=(5, 2.6))
ax[0].imshow(img);       ax[0].set_title("original");  ax[0].axis("off")
ax[1].imshow(img_baraj); ax[1].set_title("barajada");  ax[1].axis("off")
plt.tight_layout(); plt.show()


## 8. La solución: embeddings de posición

Para que el modelo sepa **dónde** estaba cada parche, se suma a cada vector un **embedding de posición**:
un vector que codifica «yo soy el parche de la fila 2, columna 3». Como los parches viven en una rejilla,
la posición es 2D (fila y columna). Lo construimos y lo miramos: ahora cada parche lleva, además de su
contenido, su **coordenada**, y barajar sí cambiaría la entrada.


In [ ]:
filas = np.arange(n*n) // n
cols  = np.arange(n*n) %  n
pos2d = np.stack([filas/(n-1), cols/(n-1)], axis=1)      # (16, 2), normalizado a [0, 1]
print("Coordenada (fila, columna) que se le suma a cada parche:")
for k in [0, 3, 5, 15]:
    print(f"  parche {k:>2} -> fila {float(pos2d[k,0]):.2f}, columna {float(pos2d[k,1]):.2f}")
print("\nAsi el modelo sabe no solo QUE hay en cada parche, sino DONDE esta.")
print("Con posicion, la imagen barajada deja de ser indistinguible: cada parche delata su sitio.")


## 9. El token de clase: cómo el ViT resume toda la imagen

Falta una pieza para clasificar. Un ViT añade al principio de la secuencia un vector extra, el **token de
clase** (`[CLS]`): no corresponde a ningún parche, es un «resumen» que, capa a capa, va recogiendo
información de todos los parches por la atención. Al final, ese único vector se usa para decidir «esto es
un gato». La secuencia pasa así de 16 a 17 elementos.


In [ ]:
cls = rng.standard_normal((1, D)) * 0.02           # vector de clase, aprendido en un modelo real
entrada_vit = np.vstack([cls, embeddings])         # [CLS] al frente + los 16 parches
print("Embeddings de parches:", embeddings.shape, " -> con [CLS] al frente:", entrada_vit.shape)
print(f"La secuencia pasa de {embeddings.shape[0]} a {entrada_vit.shape[0]} vectores: el primero es el resumen.")
print("Tras pasar por el Transformer, SOLO ese primer vector se usa para clasificar la imagen entera.")


## 10. Qué parches se parecen: la intuición de la atención

La atención compara cada parche con todos los demás. Para hacernos una idea, medimos el **parecido**
(similitud del coseno) entre los vectores de los parches: los del fondo claro se parecerán mucho entre sí;
el del círculo oscuro y el del cuadrado azul destacarán como «raros». Ese mapa de parecidos es la materia
prima con la que la atención decide a qué mirar.


In [ ]:
vn = secuencia / (np.linalg.norm(secuencia, axis=1, keepdims=True) + 1e-9)
S = vn @ vn.T                                      # matriz 16x16 de similitudes coseno
plt.figure(figsize=(4.2, 3.6))
plt.imshow(S, cmap="viridis"); plt.colorbar(label="parecido (coseno)")
plt.xticks(range(n*n)); plt.yticks(range(n*n))
plt.title("Parecido entre parches"); plt.xlabel("parche"); plt.ylabel("parche"); plt.show()

fondo = float(S[0, 3])                              # dos parches de fondo
raro  = float(S[5, 14])                             # circulo oscuro vs cuadrado azul
print(f"Parecido fondo-fondo (parches 0 y 3):           {fondo:.3f}  <- altisimo, casi iguales")
print(f"Parecido circulo-cuadrado (parches 5 y 14):     {raro:.3f}  <- bajo, son cosas distintas")
print("La atencion aprovecha justo estas diferencias para decidir a que parche prestar atencion.")


## 11. Experimento: tamaño de parche, longitud y coste

El tamaño de parche es una palanca de diseño. Parches **pequeños** dan más detalle pero una secuencia
**más larga** (y la atención cuesta del orden del cuadrado de la longitud). Parches **grandes**, lo
contrario. Lo medimos para una imagen de 224×224 (un tamaño típico), viendo cómo cambian el número de
parches y el coste relativo de la atención.


In [ ]:
print("imagen 224x224 | tam. parche | nº de parches | coste atencion (~ parches^2)")
base = None
for tam in [32, 16, 8]:
    n_p = (224 // tam) ** 2
    coste = n_p ** 2
    if base is None: base = coste
    print(f"               |    {tam:>2}x{tam:<2}    |     {n_p:>4}      |  x{coste/base:>6.1f}")
print("\nParches mas pequenos = mas detalle, pero secuencia mas larga y atencion mucho mas cara.")
print("Pasar de 32x32 a 8x8 multiplica los parches por 16 y el coste de atencion por 256.")
print("Elegir el tamano de parche es el mismo compromiso resolucion/coste de siempre.")


## 12. Verlo: la misma imagen con dos tamaños de parche

Comparamos la rejilla con `P = 16` (16 parches, gruesos) y con `P = 8` (64 parches, finos) sobre nuestra
imagen. Con `P = 8` se aprecian mejor los bordes del círculo y el cuadrado, pero la secuencia es cuatro
veces más larga. No hay un tamaño «correcto»: hay un compromiso que tú eliges.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(6, 3))
for ax, p in zip(axes, [16, 8]):
    ax.imshow(img)
    for t in range(0, H+1, p):
        ax.axhline(t-0.5, color="red", lw=0.6); ax.axvline(t-0.5, color="red", lw=0.6)
    ax.set_title(f"P={p}  ->  {(H//p)*(W//p)} parches"); ax.axis("off")
plt.tight_layout(); plt.show()
print("P=16: 16 parches gruesos.  P=8: 64 parches finos (mas detalle, secuencia 4x mas larga).")


## 13. Pruébalo tú

1. **Cambia el tamaño de parche** a `P = 8` en la celda de la sección 2: tendrás 64 parches. Vuelve a
   ejecutar y observa cómo crece la secuencia.
2. **Mete tu propia imagen** (`plt.imread`) y trocéala. Recuerda redimensionar a un múltiplo de `P`.
3. **Sube la dimensión** del embedding (`D = 64` o `128`) en la sección 6 y mira cómo cambia la forma de
   `embeddings`. ¿Qué pasaría con el coste si la subes mucho?
4. **Baraja a propósito** los parches (sección 7) y reconstruye: confirma que la «imagen» pierde sentido
   para ti pero la bolsa de vectores no cambia.
5. **Busca el parche más raro:** usando la matriz `S`, localiza el parche que menos se parece al resto.
   ¿Es el del círculo, el del cuadrado o uno de borde?


## 14. Errores comunes

- **Creer que hay «visión por computador» mágica.** No: se trocea y se aplana. Lo potente es la atención
  que viene después (facsímil 3).
- **Olvidar la posición.** Sin embeddings de posición, el modelo no sabe dónde estaba cada parche, y una
  imagen y su versión barajada son la misma bolsa de vectores: indistinguibles.
- **Olvidar el embedding lineal.** Los parches crudos pasan por una proyección aprendida antes de entrar
  al Transformer; no van «tal cual».
- **Elegir parches diminutos sin pensar.** La secuencia se dispara y la atención se vuelve carísima. Es un
  compromiso, no «más detalle siempre es mejor».
- **Confundir trocear con perder.** La partición es reversible: la imagen se reconstruye exacta. Trocear
  cambia la *forma* del dato, no su contenido.


## 15. Qué te llevas

- Una imagen entra en un Transformer como una **secuencia de parches**, cada uno convertido en un vector:
  el equivalente visual de los *tokens* de texto. Y trocear **no pierde información**.
- Por eso la **misma maquinaria** (atención, capas) sirve para texto e imagen: lo que cambia es la puerta
  de entrada, no el motor. El **embedding lineal** adapta el vector; los **embeddings de posición**
  conservan el dónde; el **token de clase** resume la imagen para decidir.
- El **tamaño de parche** es un compromiso entre detalle y coste, el mismo de siempre.

**Para seguir:** el capítulo 3 (CLIP) alinea estos vectores de imagen con vectores de texto en un mismo
espacio; el siguiente cuaderno usa una versión simple de esa idea para buscar imágenes por contenido.


---

### Ficha del cuaderno

- **Obra:** *Inteligencia artificial para gente curiosa* (facsímil interactivo).
- **Web:** https://www.iaparagentecuriosa.686f6c61.dev/
- **Autor:** @686f6c61
- **Fecha:** 2026-06-26
- **Versión:** 1.0

*Material pedagógico. Las salidas que ves son reales: se generan al ejecutar el código, no están escritas a mano. Si cambias algo, cambiarán: esa es la idea.*